# Preparing

Устанавливаем и \ или запускаем библиотеки с функциями и всякие полезные фигни. Биоблиотеки нужно установить один раз. Запускать их нужно каждый раз перед тем, как запускать код.

In [ ]:
from scipy.signal import savgol_filter
import mne #eeg analysis
import numpy as np # dataframes and staff
import matplotlib.pyplot as plt # plots
import matplotlib # plots
import PyQt5 # how plots open in you PC
from pathlib import Path # path to files
from mne.channels import find_ch_adjacency # connection btw electrodes
from functools import partial # statistics
import scipy.stats #statistics
import pickle # way to save data
import seaborn as sns # data and stats
from scipy import stats # stats
import statistics # stats
import mne_qt_browser
from mne.datasets import somato
import pathlib

#the following magic line will make all of your figures to be of the chosen size (in inches)
%matplotlib qt5

mne.viz.set_browser_backend('qt')

In [ ]:
# ДЛЯ АВТОМАТИЧЕСКОГО АНАЛИЗА ICA

import sys
!{sys.executable} -m pip install mne-icalabel
from mne_icalabel import label_components
!pip install torch

Чтоб открыть файл, нужно указать к нему путь. Путь:

In [ ]:
# для старых данных

path_to_project_old = Path('/Users/alexmedvedev/Desktop/Vishka/exp_old')

path_to_fif_saves_offset_old = Path('/Users/alexmedvedev/Desktop/Vishka/old_good_montage/fif_saves_offset')
path_to_fif_saves_tfr_offset_old = Path('/Users/alexmedvedev/Desktop/Vishka/old_good_montage/fif_saves_tfr_offset')
path_to_fif_saves_onset_old = Path('/Users/alexmedvedev/Desktop/Vishka/old_good_montage/fif_saves_onset')
path_to_fif_saves_tfr_onset_old = Path('/Users/alexmedvedev/Desktop/Vishka/old_good_montage/fif_saves_tfr_onset')

path_to_all_epochs_offset_old = Path("/Users/alexmedvedev/Desktop/Vishka/old_good_montage/all_epochs_offset")
path_to_all_epochs_offset_tfr_old = Path("/Users/alexmedvedev/Desktop/Vishka/old_good_montage/all_epochs_offset_tfr")
path_to_all_epochs_onset_old = Path("/Users/alexmedvedev/Desktop/Vishka/old_good_montage/all_epochs_onset")
path_to_all_epochs_onset_tfr_old = Path("/Users/alexmedvedev/Desktop/Vishka/old_good_montage/all_epochs_onset_tfr")

In [ ]:
# для новых данных

path_to_project_new = Path('/Users/alexmedvedev/Desktop/Vishka/exp_new')

path_to_fif_saves_new = Path('/Users/alexmedvedev/Desktop/Vishka/new/fif_saves')

path_to_all_epochs_offset_new = Path("/Users/alexmedvedev/Desktop/Vishka/new/all_epochs_offset")
path_to_all_epochs_onset_new = Path("/Users/alexmedvedev/Desktop/Vishka/new/all_epochs_onset")

# Load the file

Потом нужно открыть файл(-ы) в этом пути.

In [ ]:
sbj_id = 'AlBo' # participant's id. It's easier to load the file having this variable

In [ ]:
raw = mne.io.read_raw_brainvision(path_to_project_old / f'{sbj_id}.vhdr', eog=['HEOG', 'VEOG'], misc = ['BIP1'], preload = True)

Если в ЭЭГ есть глазные электроды, указывай это в параметре eog. misc не меняй, хз, что это, но надо. Preload позволяет обращаться к файлу со сложными задачами, без него только простые операции.

In [ ]:
raw.info

# Montage (ДЛЯ СТАРЫХ ДАННЫХ С ПЛОХИМ МОНТАЖОМ) - САШИН - используем его

In [ ]:
dict_to_rename = {
    "FCz": "C1",
    "C1": "C5",
    "C5": "TP7",
    "TP7": "CP3",
    "CP3": "P1",
    "P1": "P5",
    "P5": "PO7",
    "PO7": "PO3",
    "PO3": "POz",
    "POz": "PO4",
    "PO4": "PO8",
    "PO8": "P6",
    "P6": "P2",
    "P2": "CPz",
    "CPz": "CP4",
    "CP4": "TP8",
    "TP8": "C6",
    "C6": "C2",
    "C2": "FC4",
    "FC4": "FT8",
    "FT8": "F6",
    "F6": "AF8",
    "F2": "AF4",
    "AF4": "F2",
    "AF8": "FCz"
}

In [ ]:
try:
    raw.rename_channels (dict_to_rename)
except:
    pass
montage1020 = mne.channels.make_standard_montage('standard_1020')
for i in montage1020.ch_names:
    for j in raw.ch_names:
        if j.lower() == i. lower():
            try:
                raw.rename_channels ({j:i})
            except:
                pass
raw.set_montage('standard_1020')

# Montage (ДЛЯ СТАРЫХ ДАННЫХ С ПЛОХИМ МОНТАЖОМ) - МОЙ

Код не знает, как на голове расположены электроды. Их названия ему ничего не говорят. Надо до него это донести с мопощью подключения монтажа. Грубо говоря -- это карта расположения электродов в заивисимости от названия.

In [ ]:
# Electrode mapping
electrode_mapping = {
    "FCz": "C1",
    "C1": "C5",
    "C5": "TP7",
    "TP7": "CP3",
    "CP3": "P1",
    "P1": "P5",
    "P5": "PO7",
    "PO7": "PO3",
    "PO3": "POz",
    "POz": "PO4",
    "PO4": "PO8",
    "PO8": "P6",
    "P6": "P2",
    "P2": "CPz",
    "CPz": "CP4",
    "CP4": "TP8",
    "TP8": "C6",
    "C6": "C2",
    "C2": "FC4",
    "FC4": "FT8",
    "FT8": "F6",
    "F6": "AF8",
    "F2": "AF4",
    "AF4": "F2",
    "AF8": "FCz"
}

In [ ]:

# Step 1: Rename to temporary names
temp_mapping = {old: f"TEMP_{old}" for old in electrode_mapping if old in raw.info['ch_names']}
raw.rename_channels(temp_mapping)

In [ ]:

# Step 2: Rename to final names
final_mapping = {f"TEMP_{old}": new for old, new in electrode_mapping.items() if f"TEMP_{old}" in raw.info['ch_names']}
raw.rename_channels(final_mapping)

In [ ]:

# Set montage
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage, on_missing='ignore')

In [ ]:
raw.plot_sensors(show_names = True) #What if we would like to see the names

# Montage (ДЛЯ НОВЫХ ДАННЫХ С ХОРОШИМ МОНТАЖОМ)

Код не знает, как на голове расположены электроды. Их названия ему ничего не говорят. Надо до него это донести с мопощью подключения монтажа. Грубо говоря -- это карта расположения электродов в заивисимости от названия.

In [ ]:
montage = mne.channels.make_standard_montage("standard_1020")

In [ ]:
raw.set_montage(montage)

In [ ]:
raw.plot_sensors(show_names = True) #What if we would like to see the names

# Filtering ERP

In [ ]:
# ТОЛЬКО ДЛЯ ONSET_OLD

shift_times = np.array([0,-760,-890,-920,-760,-990,-1040,-920,-850,-890,-940,-930,-920,-830,-1000,-750,-760,-940,-960,-850,-950,-960,-970,-780,-880,-990,-930,-800,-980,-870,-840,-770,-780,-810,-810,
-830,-830,-750,-880,-910,-500,-500,-660,-620,-540,-590,-560,-620,-560,-520,-460,-600,-550,-670,-490,-460,-580,-520,-480,-450,-490,-460,-600,-660,-520,-570,-540,-540,-660,-480,-510,-440,-540,-430,-420,
-490,-460,-460,-580,-380,-760,-740,-670,-860,-740,-790,-740,-730,-850,-730,-830,-800,-950,-730,-720,-740,-730,-690,-880,-810,-770,-700,-690,-860,-770,-650,-720,-780,-780,-730,-820,-610,-730,-660,
-610,-670,-610,-670,-780,-700,-650,-520,-600,-570,-520,-700,-680,-680,-620,-810,-650,-600,-600,-700,-680,-660,-660,-510,-580,-550,-620,-680,-770,-770,-570,-540,-520,-680,-600,-660,-650,-530,-570,-510,
-560,-610,-590,-630,-640,-620,-510])

annotations = raw.annotations.copy()
shift_seconds = shift_times[:len(annotations)] / 1000
annotations.onset += shift_seconds
raw.set_annotations(annotations)

In [ ]:
raw.notch_filter(50)
raw.filter(1,45)

In [ ]:
raw.plot(duration = 100)

# Filtering TFR

In [ ]:
# ТОЛЬКО ДЛЯ ONSET_OLD

shift_times = np.array([0,-760,-890,-920,-760,-990,-1040,-920,-850,-890,-940,-930,-920,-830,-1000,-750,-760,-940,-960,-850,-950,-960,-970,-780,-880,-990,-930,-800,-980,-870,-840,-770,-780,-810,-810,
-830,-830,-750,-880,-910,-500,-500,-660,-620,-540,-590,-560,-620,-560,-520,-460,-600,-550,-670,-490,-460,-580,-520,-480,-450,-490,-460,-600,-660,-520,-570,-540,-540,-660,-480,-510,-440,-540,-430,-420,
-490,-460,-460,-580,-380,-760,-740,-670,-860,-740,-790,-740,-730,-850,-730,-830,-800,-950,-730,-720,-740,-730,-690,-880,-810,-770,-700,-690,-860,-770,-650,-720,-780,-780,-730,-820,-610,-730,-660,
-610,-670,-610,-670,-780,-700,-650,-520,-600,-570,-520,-700,-680,-680,-620,-810,-650,-600,-600,-700,-680,-660,-660,-510,-580,-550,-620,-680,-770,-770,-570,-540,-520,-680,-600,-660,-650,-530,-570,-510,
-560,-610,-590,-630,-640,-620,-510])

annotations = raw.annotations.copy()
shift_seconds = shift_times[:len(annotations)] / 1000
annotations.onset += shift_seconds
raw.set_annotations(annotations)

In [ ]:
raw.notch_filter(105)
raw.filter(1,100)

In [ ]:
raw.plot(duration = 100)

# ICA

Individual Component Analysis "объясняет" данные записи, деля их на компоненты. Например, фронтальный компонент (глаза), затылочный, и тд.
Это нужно для извлечения морганий.

In [ ]:
ica = mne.preprocessing.ICA(n_components = 0.99)

In [ ]:
ica.fit(raw, reject_by_annotation=True)

In [ ]:
ica.plot_sources(raw)

In [ ]:
ica.plot_components()

In [ ]:
ic_labels = label_components(raw, ica, method="iclabel")

# Automatic detection - adjust probabilities as needed
bad_components = [
    i for i, (label, proba) in enumerate(zip(ic_labels["labels"], ic_labels["y_pred_proba"]))
    if (label in ["eye blink", "muscle activity", "channel noise", "heart beat"]) 
    and (proba > 0.5)  # Confidence threshold
]

print(f"Automatically detected bad components: {bad_components}")

In [ ]:
ica.exclude = [1, 2, 12, 13, 15, 28]
ica.apply(raw)

In [ ]:
raw.plot(duration = 100)

In [ ]:
# ERP_offset
raw.save(path_to_fif_saves_offset_old/ f'{sbj_id}_eeg.fif', overwrite=True) #saving preprocessed file

In [ ]:
# ERP_onset
raw.save(path_to_fif_saves_onset_old/ f'{sbj_id}_eeg.fif', overwrite=True)

In [ ]:
# TFR_offset
raw.save(path_to_fif_saves_tfr_offset_old/ f'{sbj_id}_eeg.fif', overwrite=True) #saving preprocessed file

In [ ]:
# TFR_onset
raw.save(path_to_fif_saves_tfr_onset_old/ f'{sbj_id}_eeg.fif', overwrite=True)

# Epochs prepare

Эпохи -- это вычленение из записи только секунды после метки.

In [ ]:
events, ids = mne.events_from_annotations(raw)

In [ ]:
ids

In [ ]:
all_ids = {'5-6': 5 or 51, '7-8': 7 or 71, '9-10': 9 or 91, '11-12': 11 or 111, 'contr_5-6': 25, 'contr_7-8': 27, 'contr_9-10': 29 or 35, 'contr_11-12': 211 or 63}

In [ ]:
all_events = mne.pick_events(events, include = [5,51,7,71,9,91,11,111,25,251,27,271,29,35,211,63])

tmin -- престимульный интервал, часто 0.5 - 0.1. tmax -- постстимульный интервал -- чаще 1. baseline первое число совпадает с престимулом, второе число 0.

# Epochs old offset ERP

In [ ]:
mne.viz.plot_events(all_events, event_id = all_ids, sfreq = raw.info['sfreq'])
epochs_dis = mne.Epochs(raw, all_events, all_ids, tmin = -0.2, tmax = 1, baseline = (-0.2, 0), reject_by_annotation=True)
epochs_dis.plot()

In [ ]:
epochs_dis.save(path_to_all_epochs_offset_old/ f'{sbj_id}_epo.fif', overwrite=True)

# Epochs old onset ERP

In [ ]:
mne.viz.plot_events(all_events, event_id = all_ids, sfreq = raw.info['sfreq'])
epochs_dis = mne.Epochs(raw, all_events, all_ids, tmin = -0.2, tmax = 2, baseline = (-0.2, 0), reject_by_annotation=True)
epochs_dis.plot()

In [ ]:
epochs_dis.save(path_to_all_epochs_onset_old/ f'{sbj_id}_epo.fif', overwrite=True)

# Epochs old offset TFR

In [ ]:
mne.viz.plot_events(all_events, event_id = all_ids, sfreq = raw.info['sfreq'])
epochs_dis = mne.Epochs(raw, all_events, all_ids, tmin = -0.2, tmax = 1, baseline = (-0.2, 0), reject_by_annotation=True)
epochs_dis.plot()

In [ ]:
epochs_dis.save(path_to_all_epochs_offset_tfr_old/ f'{sbj_id}_epo.fif', overwrite=True)

# Epochs old onset TFR

In [ ]:

mne.viz.plot_events(all_events, event_id = all_ids, sfreq = raw.info['sfreq'])
epochs_dis = mne.Epochs(raw,all_events,all_ids,tmin=-0.2,tmax=2, baseline = (-0.2,0),reject_by_annotation = True)
epochs_dis.plot()

In [ ]:
epochs_dis.save(path_to_all_epochs_onset_tfr_old/ f'{sbj_id}_epo.fif', overwrite=True)

# Epochs new offset

In [ ]:
mne.viz.plot_events(all_events, event_id = all_ids, sfreq = raw.info['sfreq'])
epochs_dis = mne.Epochs(raw, all_events, all_ids, tmin = -0.2, tmax = 1, baseline = (-0.2, 0), reject_by_annotation=True)
epochs_dis.plot()

In [ ]:
epochs_dis.save(path_to_all_epochs_offset_new/ f'{sbj_id}_epo.fif', overwrite=True)

# Epochs new onset (НЕ ДОДЕЛАНО)

In [ ]:
shift_times = np.array([-760,-890,-920,-760,-990,-1040,-920,-850,-890,-940,-930,-920,-830,-1000,-750,-760,-940,-960,-850,-950,-960,-970,-780,-880,-990,-930,-800,-980,-870,-840,-770,-780,-810,-810,
-830,-830,-750,-880,-910,-500,-500,-660,-620,-540,-590,-560,-620,-560,-520,-460,-600,-550,-670,-490,-460,-580,-520,-480,-450,-490,-460,-600,-660,-520,-570,-540,-540,-660,-480,-510,-440,-540,-430,-420,
-490,-460,-460,-580,-380,-760,-740,-670,-860,-740,-790,-740,-730,-850,-730,-830,-800,-950,-730,-720,-740,-730,-690,-880,-810,-770,-700,-690,-860,-770,-650,-720,-780,-780,-730,-820,-610,-730,-660,
-610,-670,-610,-670,-780,-700,-650,-520,-600,-570,-520,-700,-680,-680,-620,-810,-650,-600,-600,-700,-680,-660,-660,-510,-580,-550,-620,-680,-770,-770,-570,-540,-520,-680,-600,-660,-650,-530,-570,-510,
-560,-610,-590,-630,-640,-620,-510]) # ДОБАВИТЬ ДЛЯ НОВЫХ КОНТРОЛЬНЫХ ЭПОХ

In [ ]:
events = np.array(all_events)
sfreq = raw.info['sfreq']
shift_samples = (shift_times/1000)*sfreq

In [ ]:
shifted_events = np.copy(all_events)

In [ ]:
shifted_events[:,0] += shift_samples.astype(int)
epochs_dis = mne.Epochs(raw,all_events,all_ids,tmin=-0.2,tmax=2, baseline = (-0.2,0),reject_by_annotation = True)
epochs_dis.plot()

In [ ]:
epochs_dis.save(path_to_all_epochs_onset_new/ f'{sbj_id}_epo.fif', overwrite=True)

# Evokeds

ERP из эпох.

In [ ]:
evoked_dis = epochs_dis.average()
evoked_dis

In [ ]:
evoked_dis.plot()

In [ ]:
my_evokeds = {}
for condition in epochs_dis.event_id:
    my_evokeds[condition] =  epochs_dis[condition].average()
    
my_evokeds

In [ ]:
my_evokeds['11-12'].plot_topo() # условие в квадратной скобке можно менять

In [ ]:
mne.viz.plot_compare_evokeds(my_evokeds)

In [ ]:
mne.viz.plot_compare_evokeds(
    my_evokeds,
    axes="topo",
    vlines = [0.0, 0.1, 0.3]
) #useless

# ERP

In [ ]:
import pickle

In [ ]:
all_dis_2 = [file for file in path_to_all_epochs_offset_old.iterdir() if file.name.endswith('fif')]
all_dis_2

In [ ]:
all_dis_epochs_2 = {file.name: None for file in all_dis_2}

for file in all_dis_2:  
    dis_epochs_2 = mne.read_epochs(file)
    # dis_epochs_2 = dis_epochs_2 ['11-12'] # если нужны конкретные
    all_dis_epochs_2[file.name] = dis_epochs_2

In [ ]:
all_dis_erps_2 = {name : {} for name in all_dis_epochs_2}
for name in all_dis_epochs_2:
    for event_id in all_dis_epochs_2[name].event_id:
        current_dis_epochs_2 = all_dis_epochs_2[name][event_id]
        current_dis_epochs_2.savgol_filter(10.)
        dis_erp_2 = current_dis_epochs_2.copy().load_data().interpolate_bads().average()
        all_dis_erps_2[name][event_id] = dis_erp_2

In [ ]:
all_dis_erps_2

In [ ]:
grand_averages_2 = {event_id : mne.grand_average([evokeds[event_id] for evokeds in all_dis_erps_2.values()], drop_bads = False) for event_id in [[key for key in val.keys()] for ival, val in enumerate(all_dis_erps_2.values()) if ival == 0][0]}

## Plots

In [ ]:
mne.viz.plot_compare_evokeds({event_id : ga.copy().pick_channels(['C1', 'Cz', 'C2', 'FC1', 'FCz','FC2', 'F1', 'Fz', 'F2']) for event_id, ga in grand_averages_2.items()}, combine = 'mean')

In [ ]:
mne.viz.plot_compare_evokeds(
    {event_id : ga.copy() for event_id, ga in grand_averages_2.items()},
    colors={'5-6' : 0, '7-8' : 1, '9-10':2, '11-12':3},
    linestyles={'5-6' : "solid", '7-8' : "solid", '9-10':"solid", '11-12':"solid"},
    axes="topo",
    vlines = [0.0, 0.1, 0.3, 0.4]
)

In [ ]:
grand_averages_2["11-12"].plot_joint()

In [ ]:
times = np.arange(0.05, 0.65, 0.05)

In [ ]:
grand_averages_2['11-12'].plot_topomap(times = times)